# Batch Prediction for Test Set



## Enter the MMSegmentation main directory

In [1]:
import os
os.chdir('mmsegmentation')

In [2]:
os.getcwd()

'/home/featurize/work/Mask2Former/mmsegmentation'

## Import Packages

In [3]:
import os
import numpy as np
import cv2
from tqdm import tqdm

from mmseg.apis import init_model, inference_model, show_result_pyplot
import mmcv

import matplotlib.pyplot as plt
%matplotlib inline

/environment/miniconda3/envs/py38/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Model

In [4]:
# 模型 config 配置文件
config_file = '../WeightandCode/trans-fibre/ZihaoDataset_Mask2Former_20230818.py'

checkpoint_file = '../WeightandCode/trans-fibre/fibre_Mask2Former.pth'

# 计算硬件
# device = 'cpu'
device = 'cuda:0'

In [5]:
model = init_model(config_file, checkpoint_file, device=device)

Loads checkpoint by local backend from path: ../WeightandCode/trans-fibre/fibre_Mask2Former.pth


## Specify color schemes for each category

In [6]:
# 每个类别的 BGR 配色
palette = [
    ['background', [0,0,0]],
    ['fibre', [255,255,255]]         #Xylem Vessel fibre ray-s代表横切    ray代表弦切 0 0 127
]

palette_dict = {}
for idx, each in enumerate(palette):
    palette_dict[idx] = each[1]

In [7]:
palette_dict

{0: [0, 0, 0], 1: [255, 255, 255]}

## Specify the test set path (can also be changed to the folder path of images to be tested)

In [8]:
PATH_IMAGE = '../Input'

In [9]:
os.chdir(PATH_IMAGE)

## Single Image Prediction Function

In [10]:
opacity = 0 # 透明度，越大越接近原图

In [11]:
def process_single_img(img_path, save=False):
    
    img_bgr = cv2.imread(img_path)

    # 语义分割预测
    result = inference_model(model, img_bgr)
    pred_mask = result.pred_sem_seg.data[0].cpu().numpy()

    # 将预测的整数ID，映射为对应类别的颜色
    pred_mask_bgr = np.zeros((pred_mask.shape[0], pred_mask.shape[1], 3))
    for idx in palette_dict.keys():
        pred_mask_bgr[np.where(pred_mask==idx)] = palette_dict[idx]
    pred_mask_bgr = pred_mask_bgr.astype('uint8')

    # 将语义分割预测图和原图叠加显示
    pred_viz = cv2.addWeighted(img_bgr, opacity, pred_mask_bgr, 1-opacity, 0)
    
    # 保存图像至 outputs/testset-pred 目录
    if save:
        save_path = os.path.join('../Output/'+img_path.split('/')[-1])
        cv2.imwrite(save_path, pred_viz)


## Batch Prediction for Test Set

In [12]:
for each in tqdm(os.listdir()):
    process_single_img(each, save=True)

  0%|          | 0/32 [00:00<?, ?it/s]/environment/miniconda3/envs/py38/lib/python3.8/site-packages/mmdet/models/layers/positional_encoding.py:103: UserWarning: __floordiv__ is deprecated, and its behavior will change in a future version of pytorch. It currently rounds toward 0 (like the 'trunc' function NOT 'floor'). This results in incorrect rounding for negative values. To keep the current behavior, use torch.div(a, b, rounding_mode='trunc'), or for actual floor division, use torch.div(a, b, rounding_mode='floor').
  dim_t = self.temperature**(2 * (dim_t // 2) / self.num_feats)
/environment/miniconda3/envs/py38/lib/python3.8/site-packages/torch/functional.py:445: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at  ../aten/src/ATen/native/TensorShape.cpp:2157.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|██████████| 32/32 [00:36<00:00,  1.14s/it]


预测结果保存在 `mmsegmentation/outputs/testset-pred` 目录下

In [14]:
os.chdir(os.path.join('../','../','../'))

## Delete unnecessary files automatically generated by the system

### View redundant files to be deleted

In [55]:
!find . -iname '__MACOSX'

In [56]:
!find . -iname '.DS_Store'

In [57]:
!find . -iname '.ipynb_checkpoints'

./.ipynb_checkpoints


### Delete unnecessary files

In [58]:
!for i in `find . -iname '__MACOSX'`; do rm -rf $i;done

In [59]:
!for i in `find . -iname '.DS_Store'`; do rm -rf $i;done

In [60]:
!for i in `find . -iname '.ipynb_checkpoints'`; do rm -rf $i;done

### Verify that redundant files have been deleted

In [ ]:
!find . -iname '__MACOSX'

In [ ]:
!find . -iname '.DS_Store'

In [ ]:
!find . -iname '.ipynb_checkpoints'

### Calculate the area under a 10X magnification scope

In [25]:
import os
import numpy as np
import pandas as pd
from PIL import Image


def count_target_pixels(image_path, target_color, tolerance=0):
    """
    统计图像中目标颜色的像素数量
    """
    img = Image.open(image_path).convert('RGBA')
    image_array = np.array(img)

    target_red, target_green, target_blue = target_color

    target_pixels = np.sum(
        (image_array[:, :, 0] >= target_red - tolerance) & (image_array[:, :, 0] <= target_red + tolerance) &
        (image_array[:, :, 1] >= target_green - tolerance) & (image_array[:, :, 1] <= target_green + tolerance) &
        (image_array[:, :, 2] >= target_blue - tolerance) & (image_array[:, :, 2] <= target_blue + tolerance)
    )

    return int(target_pixels)


def calculate_percentage(target_pixels, total_pixels):
    """
    计算目标颜色的像素占比
    """
    if total_pixels == 0:
        return 0.0
    return (target_pixels / total_pixels) * 100.0


def process_folder_percentage(
    input_folder,
    output_csv,
    target_color,
    tolerance=0,
    pixel_to_cm_ratio=0.0188686,
    scale_factor=1.387
):
    """
    处理文件夹中的图片，计算目标颜色占整张图片的面积百分比，保存到 CSV 文件。
    并将面积单位换算成实际的真实面积（平方厘米）。

    Parameters:
        input_folder (str): 输入文件夹路径
        output_csv (str): 输出 CSV 文件路径
        target_color (list): 目标颜色 [R, G, B]
        tolerance (int): 颜色容差范围，默认值为 0
        pixel_to_cm_ratio (float): 像素到厘米的换算比例
        scale_factor (float): 你原代码里对 Target Pixels / Area / Percentage 乘的系数（默认 1.131）

    Returns:
        None
    """
    results = []

    # 每个像素对应的面积（单位：平方厘米）
    pixel_area_cm2 = pixel_to_cm_ratio ** 2

    # 若输出目录不存在，自动创建
    out_dir = os.path.dirname(output_csv)
    if out_dir and (not os.path.exists(out_dir)):
        os.makedirs(out_dir, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            image_path = os.path.join(input_folder, filename)

            try:
                # 打开图像并获取分辨率
                img = Image.open(image_path)
                image_width, image_height = img.size
                total_pixels = int(image_width * image_height)

                # 统计目标颜色的像素数量
                target_pixels = count_target_pixels(image_path, target_color, tolerance)

                # 计算目标颜色的百分比
                percentage = calculate_percentage(target_pixels, total_pixels)

                # 将目标像素数转换为真实面积（单位：平方厘米）
                real_area_cm2 = target_pixels * pixel_area_cm2

                # 保存结果（保持你原来的 1.131 逻辑）
                results.append({
                    'Image Name': filename,
                    'Image Width (pixels)': image_width,
                    'Image Height (pixels)': image_height,
                    'Total Pixels': total_pixels,
                    'Target Pixels': target_pixels * scale_factor * 1.387,
                    'Real Area (cm^2)': real_area_cm2 * scale_factor * 1.387,   # 真实面积（平方厘米）
                    'Percentage (%)': percentage * scale_factor * 1.387
                })

            except Exception as e:
                print(f"无法处理图片 {filename}: {e}")

    df = pd.DataFrame(results)

    # 保存结果到 CSV（utf-8-sig：Excel 打开中文更不容易乱码）
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"结果已保存到 {output_csv}")


# ===== 示例使用 =====
if __name__ == "__main__":
    input_folder = r'../Output'                 # 输入图片文件夹路径
    output_csv = r'../Output/10xFiber.csv'    # 输出 CSV 文件路径
    target_color = [255, 255, 255]              # 目标颜色 (白色)
    tolerance = 2                               # 容差范围
    pixel_to_cm_ratio = 100 / 293               # 20x为50：293（按你原注释保留）
    process_folder_percentage(input_folder, output_csv, target_color, tolerance, pixel_to_cm_ratio)


结果已保存到 ../Output/10xVesssel.csv


### Calculate the area under a 20X magnification scope

In [13]:
import os
import numpy as np
import pandas as pd
from PIL import Image


def count_target_pixels(image_path, target_color, tolerance=0):
    """
    统计图像中目标颜色的像素数量
    """
    img = Image.open(image_path).convert('RGBA')
    image_array = np.array(img)

    target_red, target_green, target_blue = target_color

    target_pixels = np.sum(
        (image_array[:, :, 0] >= target_red - tolerance) & (image_array[:, :, 0] <= target_red + tolerance) &
        (image_array[:, :, 1] >= target_green - tolerance) & (image_array[:, :, 1] <= target_green + tolerance) &
        (image_array[:, :, 2] >= target_blue - tolerance) & (image_array[:, :, 2] <= target_blue + tolerance)
    )

    return int(target_pixels)


def calculate_percentage(target_pixels, total_pixels):
    """
    计算目标颜色的像素占比
    """
    if total_pixels == 0:
        return 0.0
    return (target_pixels / total_pixels) * 100.0


def process_folder_percentage(
    input_folder,
    output_csv,
    target_color,
    tolerance=0,
    pixel_to_cm_ratio=0.0188686,
    scale_factor=1.387
):
    """
    处理文件夹中的图片，计算目标颜色占整张图片的面积百分比，保存到 CSV 文件。
    并将面积单位换算成实际的真实面积（平方厘米）。

    Parameters:
        input_folder (str): 输入文件夹路径
        output_csv (str): 输出 CSV 文件路径
        target_color (list): 目标颜色 [R, G, B]
        tolerance (int): 颜色容差范围，默认值为 0
        pixel_to_cm_ratio (float): 像素到厘米的换算比例
        scale_factor (float): 你原代码里对 Target Pixels / Area / Percentage 乘的系数（默认 1.131）

    Returns:
        None
    """
    results = []

    # 每个像素对应的面积（单位：平方厘米）
    pixel_area_cm2 = pixel_to_cm_ratio ** 2

    # 若输出目录不存在，自动创建
    out_dir = os.path.dirname(output_csv)
    if out_dir and (not os.path.exists(out_dir)):
        os.makedirs(out_dir, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            image_path = os.path.join(input_folder, filename)

            try:
                # 打开图像并获取分辨率
                img = Image.open(image_path)
                image_width, image_height = img.size
                total_pixels = int(image_width * image_height)

                # 统计目标颜色的像素数量
                target_pixels = count_target_pixels(image_path, target_color, tolerance)

                # 计算目标颜色的百分比
                percentage = calculate_percentage(target_pixels, total_pixels)

                # 将目标像素数转换为真实面积（单位：平方厘米）
                real_area_cm2 = target_pixels * pixel_area_cm2

                # 保存结果（保持你原来的 1.131 逻辑）
                results.append({
                    'Image Name': filename,
                    'Image Width (pixels)': image_width,
                    'Image Height (pixels)': image_height,
                    'Total Pixels': total_pixels,
                    'Target Pixels': target_pixels * scale_factor * 1.387,
                    'Real Area (cm^2)': real_area_cm2 * scale_factor * 1.387,   # 真实面积（平方厘米）
                    'Percentage (%)': percentage * scale_factor * 1.387
                })

            except Exception as e:
                print(f"无法处理图片 {filename}: {e}")

    df = pd.DataFrame(results)

    # 保存结果到 CSV（utf-8-sig：Excel 打开中文更不容易乱码）
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"结果已保存到 {output_csv}")


# ===== 示例使用 =====
if __name__ == "__main__":
    input_folder = r'../Output'                 # 输入图片文件夹路径
    output_csv = r'../Output/20xFiber.csv'    # 输出 CSV 文件路径
    target_color = [255, 255, 255]              # 目标颜色 (白色)
    tolerance = 2                               # 容差范围
    pixel_to_cm_ratio = 50 / 293               # 20x为50：293（按你原注释保留）
    process_folder_percentage(input_folder, output_csv, target_color, tolerance, pixel_to_cm_ratio)


结果已保存到 ../Output/20xFiber.csv


### Calculate the area under a 40X magnification scope

In [27]:
import os
import numpy as np
import pandas as pd
from PIL import Image


def count_target_pixels(image_path, target_color, tolerance=0):
    """
    统计图像中目标颜色的像素数量
    """
    img = Image.open(image_path).convert('RGBA')
    image_array = np.array(img)

    target_red, target_green, target_blue = target_color

    target_pixels = np.sum(
        (image_array[:, :, 0] >= target_red - tolerance) & (image_array[:, :, 0] <= target_red + tolerance) &
        (image_array[:, :, 1] >= target_green - tolerance) & (image_array[:, :, 1] <= target_green + tolerance) &
        (image_array[:, :, 2] >= target_blue - tolerance) & (image_array[:, :, 2] <= target_blue + tolerance)
    )

    return int(target_pixels)


def calculate_percentage(target_pixels, total_pixels):
    """
    计算目标颜色的像素占比
    """
    if total_pixels == 0:
        return 0.0
    return (target_pixels / total_pixels) * 100.0


def process_folder_percentage(
    input_folder,
    output_csv,
    target_color,
    tolerance=0,
    pixel_to_cm_ratio=0.0188686,
    scale_factor=1.387
):
    """
    处理文件夹中的图片，计算目标颜色占整张图片的面积百分比，保存到 CSV 文件。
    并将面积单位换算成实际的真实面积（平方厘米）。

    Parameters:
        input_folder (str): 输入文件夹路径
        output_csv (str): 输出 CSV 文件路径
        target_color (list): 目标颜色 [R, G, B]
        tolerance (int): 颜色容差范围，默认值为 0
        pixel_to_cm_ratio (float): 像素到厘米的换算比例
        scale_factor (float): 你原代码里对 Target Pixels / Area / Percentage 乘的系数（默认 1.131）

    Returns:
        None
    """
    results = []

    # 每个像素对应的面积（单位：平方厘米）
    pixel_area_cm2 = pixel_to_cm_ratio ** 2

    # 若输出目录不存在，自动创建
    out_dir = os.path.dirname(output_csv)
    if out_dir and (not os.path.exists(out_dir)):
        os.makedirs(out_dir, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            image_path = os.path.join(input_folder, filename)

            try:
                # 打开图像并获取分辨率
                img = Image.open(image_path)
                image_width, image_height = img.size
                total_pixels = int(image_width * image_height)

                # 统计目标颜色的像素数量
                target_pixels = count_target_pixels(image_path, target_color, tolerance)

                # 计算目标颜色的百分比
                percentage = calculate_percentage(target_pixels, total_pixels)

                # 将目标像素数转换为真实面积（单位：平方厘米）
                real_area_cm2 = target_pixels * pixel_area_cm2

                # 保存结果（保持你原来的 1.131 逻辑）
                results.append({
                    'Image Name': filename,
                    'Image Width (pixels)': image_width,
                    'Image Height (pixels)': image_height,
                    'Total Pixels': total_pixels,
                    'Target Pixels': target_pixels * scale_factor * 1.387,
                    'Real Area (cm^2)': real_area_cm2 * scale_factor * 1.387,   # 真实面积（平方厘米）
                    'Percentage (%)': percentage * scale_factor * 1.387
                })

            except Exception as e:
                print(f"无法处理图片 {filename}: {e}")

    df = pd.DataFrame(results)

    # 保存结果到 CSV（utf-8-sig：Excel 打开中文更不容易乱码）
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"结果已保存到 {output_csv}")


# ===== 示例使用 =====
if __name__ == "__main__":
    input_folder = r'../Output'                 # 输入图片文件夹路径
    output_csv = r'../Output/40xFiber.csv'    # 输出 CSV 文件路径
    target_color = [255, 255, 255]              # 目标颜色 (白色)
    tolerance = 2                               # 容差范围
    pixel_to_cm_ratio = 20 / 234               # 20x为50：293（按你原注释保留）
    process_folder_percentage(input_folder, output_csv, target_color, tolerance, pixel_to_cm_ratio)


结果已保存到 ../Output/40xVesssel.csv
